# OpenKapitza on a Colab GPU: MACE in LAMMPS (test run)

What this notebook does, on the **free T4** runtime (Runtime → Change runtime type → T4 GPU):

1. builds LAMMPS stable_30Sep2026 with KOKKOS (CUDA), ML-IAP and PYTHON, and caches the build on your Google Drive, so only the first session pays for it (expect **45–90 min** the first time, ~3 min after);
2. converts the **MACE-MP-0 small** foundation model to LAMMPS's ML-IAP format;
3. checks it on bulk Si, then runs the **Si/Ge NEMD deck** (campaign cell r00, 35k atoms) for a few hundred steps per stage with the model: does it run end to end, and how fast;
4. times the same cell with **Tersoff on the GPU** (KOKKOS), for comparison with the CPU runs.

It ends with a summary to paste back into the chat. Everything here is a plumbing and timing test: MACE-MP-0 has no Si/Ge interface in its training set and is ungated, so **no h_K from it is quotable** (the job's flags say so).

**Status: written without a GPU to test on.** Each step checks its own result and stops with a clear message; if one fails, paste the message back. Run the cells in order. If the session disconnects, run them again from the top: finished steps are reused from Drive.

In [ ]:
# 1. GPU, Drive, paths
import os, subprocess, json, time, shutil, glob, pathlib
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert out.returncode == 0, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
GPU = out.stdout.strip(); print("GPU:", GPU)
CC = GPU.split(",")[-1].strip().replace(".", "")          # e.g. 7.5 -> 75
ARCH = {"75": "TURING75", "80": "AMPERE80", "86": "AMPERE86", "89": "ADA89", "90": "HOPPER90"}.get(CC)
assert ARCH, f"compute capability {CC}: add its Kokkos_ARCH name to this cell"

from google.colab import drive
drive.mount("/content/drive")
CACHE = pathlib.Path("/content/drive/MyDrive/openkapitza_colab"); CACHE.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path("/content/ok"); WORK.mkdir(exist_ok=True)
TAG, COMMIT = "stable_30Sep2026", "8de817dd79bfe4525d5d39246a212d833e6dee07"
SUMMARY = {"gpu": GPU, "kokkos_arch": ARCH}
def sh(cmd, check=True, **kw):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f"failed: {cmd}")
    return r
print("cache:", CACHE)

In [ ]:
# 2. Python packages (MACE, cuEquivariance, CuPy for ML-IAP on KOKKOS, Cython for the LAMMPS build)
sh("pip install -q mace-torch cython cupy-cuda12x cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12")
import torch; print("torch", torch.__version__, "CUDA", torch.version.cuda, "GPU ok:", torch.cuda.is_available())
SUMMARY["torch"] = torch.__version__

In [ ]:
# 3. Sources: OpenKapitza and LAMMPS at the validated commit
os.chdir(WORK)
if not (WORK / "OpenKapitza").exists():
    sh("git clone -q https://github.com/ariahosseini/OpenKapitza.git")
sh("git -C OpenKapitza pull -q")
sh("pip install -q -e OpenKapitza")
if not (WORK / "lammps").exists():
    sh(f"git clone -q --depth 1 --branch {TAG} https://github.com/lammps/lammps.git")
head = sh("git -C lammps rev-parse HEAD").stdout.strip()
assert head == COMMIT, f"LAMMPS at {head}, expected {COMMIT}"
print("OpenKapitza", sh("git -C OpenKapitza log -1 --format=%h").stdout.strip(), "| LAMMPS", TAG)

In [ ]:
# 4. Build LAMMPS (KOKKOS-CUDA + ML-IAP + PYTHON), or restore it from Drive
import sys
BUILD = WORK / "lammps" / "build-gpu"
TARBALL = CACHE / f"lammps-{TAG}-{ARCH}.tar.gz"
if TARBALL.exists() and not BUILD.exists():
    print("restoring the build from Drive"); sh(f"tar xzf {TARBALL} -C {WORK / 'lammps'}")
if not (BUILD / "lmp").exists():
    BUILD.mkdir(exist_ok=True); os.chdir(BUILD)
    flags = " ".join([
        "-C ../cmake/presets/kokkos-cuda.cmake", "-D CMAKE_BUILD_TYPE=Release", "-D BUILD_MPI=OFF",
        f"-D Kokkos_ARCH_{ARCH}=ON", "-D BUILD_SHARED_LIBS=ON", "-D CMAKE_INSTALL_PREFIX=" + str(BUILD),
        "-D PKG_ML-IAP=ON", "-D PKG_ML-SNAP=ON", "-D PKG_PYTHON=ON", "-D MLIAP_ENABLE_PYTHON=ON",
        "-D PKG_MANYBODY=ON", "-D PKG_EXTRA-FIX=ON", "-D PKG_EXTRA-COMPUTE=ON",
        f"-D Python_EXECUTABLE={sys.executable}"])
    sh(f"cmake {flags} ../cmake > cmake.log 2>&1")
    t0 = time.time()
    r = sh("cmake --build . -j 4 > build.log 2>&1", check=False)
    if r.returncode != 0:
        print(open("build.log").read()[-4000:]); raise RuntimeError("build failed (end of build.log above)")
    print(f"built in {(time.time() - t0) / 60:.0f} min; caching to Drive")
    sh(f"tar czf {TARBALL} -C {WORK / 'lammps'} build-gpu")
os.chdir(BUILD)
sh("make install-python > install-python.log 2>&1")
LMP = str(BUILD / "lmp")
os.environ["LD_LIBRARY_PATH"] = f"{BUILD}:" + os.environ.get("LD_LIBRARY_PATH", "")
help_ = sh(f"{LMP} -h").stdout
pk = help_.split("Installed packages:")[1].split("List of individual")[0].split()
print("packages:", " ".join(pk))
for p in ("KOKKOS", "ML-IAP", "PYTHON", "MANYBODY"):
    assert p in pk, f"{p} missing from the build"
SUMMARY["lammps_packages"] = pk
KK = "-k on g 1 -sf kk -pk kokkos newton on neigh half"

In [ ]:
# 5. MACE-MP-0 small -> ML-IAP model file (converted on this GPU, as the MACE docs require)
MODEL_DIR = WORK / "models"; MODEL_DIR.mkdir(exist_ok=True)
target = MODEL_DIR / "mace_mp0_small.model-mliap_lammps.pt"
cached = CACHE / f"{target.name}.{ARCH}"
if cached.exists():
    shutil.copy(cached, target)
else:
    try:                                                   # recent mace-torch
        from mace.calculators.foundations_models import download_mace_mp_checkpoint
        src = download_mace_mp_checkpoint("small")
    except ImportError:                                    # older: download via the calculator
        from mace.calculators import mace_mp
        mace_mp(model="small", device="cuda", default_dtype="float64")
        src = sorted(glob.glob(os.path.expanduser("~/.cache/mace/*.model")), key=os.path.getmtime)[-1]
    print("checkpoint:", src)
    shutil.copy(src, MODEL_DIR / "mace_mp0_small.model")
    os.chdir(MODEL_DIR)
    sh("mace_create_lammps_model mace_mp0_small.model --format=mliap")
    assert target.exists(), f"no {target.name}; files: {os.listdir(MODEL_DIR)}"
    shutil.copy(target, cached)
print("model:", target, f"{target.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 6. Sanity check: bulk Si (512 atoms) with MACE on the GPU
d = WORK / "bulk_si"; d.mkdir(exist_ok=True); shutil.copy(target, d)
(d / "in.si").write_text(f'''units metal
atom_style atomic
newton on
lattice diamond 5.431
region box block 0 4 0 4 0 4
create_box 1 box
create_atoms 1 box
mass 1 28.0855
pair_style mliap unified {target.name} 0
pair_coeff * * Si
velocity all create 300 4711
fix 1 all nve
thermo 50
thermo_style custom step temp pe press
run 200
variable e equal pe/atoms
print "E_PER_ATOM ${{e}}"
''')
os.chdir(d)
r = sh(f"{LMP} {KK} -in in.si -log log.si", check=False)
log = (d / "log.si").read_text() if (d / "log.si").exists() else r.stdout
if "E_PER_ATOM" not in log:
    print(log[-3000:], r.stderr[-2000:]); raise RuntimeError("bulk Si with MACE did not run (log above)")
e = float(log.split("E_PER_ATOM")[1].split()[0])
print(f"energy {e:.3f} eV/atom (MACE-MP-0 is fitted to PBE; Materials Project Si is about -5.42)")
SUMMARY["bulk_si_eV_per_atom"] = e

In [ ]:
# 7. The Si/Ge NEMD deck with MACE, every stage cut to a few hundred steps
os.chdir(WORK / "OpenKapitza")
JOBS = WORK / "jobs"
sh(f"python3 scripts/prepare_mlip.py {JOBS} --model examples/mlip/mace_mp0_small.yaml "
   f"--model-dir {MODEL_DIR} --gpu --smoke --name sige_mace_mp0")
wd = JOBS / "mlip" / "sige_mace_mp0"
os.chdir(wd); t0 = time.time()
r = sh(f"{LMP} {KK} -in nemd.in -log log.lammps > stdout.log 2> stderr.log", check=False)
secs = time.time() - t0
sys.path.insert(0, str(WORK / "OpenKapitza" / "scripts"))
import run_smoke
probs = run_smoke.check(wd); sp = run_smoke.speed(wd)
print("exit", r.returncode, f"{secs / 60:.1f} min;", "PASS" if not probs else "FAIL: " + "; ".join(probs))
if sp:
    full = json.load(open(JOBS / "manifest.json"))["jobs"][-1]
    rate = sp["rate"]                                     # atom-steps per second on one GPU
    r00_steps = 750_000                                   # the full r00 deck: relax + eq + steady + production
    hours = full["atoms"] * r00_steps / rate / 3600
    print(f"MACE-MP-0 small: {rate:.3g} atom-steps/s -> the full r00 run would take ~{hours:.0f} h on this GPU")
    SUMMARY["mace_sige"] = {"ok": not probs, "problems": probs, "atom_steps_per_s": rate,
                            "full_r00_hours": round(hours, 1), "atoms": full["atoms"]}
else:
    print(open(wd / "log.lammps").read()[-3000:] if (wd / "log.lammps").exists() else r.stderr[-3000:])
    SUMMARY["mace_sige"] = {"ok": False, "problems": probs}

In [ ]:
# 8. Tersoff on the GPU, same cell, for comparison with the CPU runs
os.chdir(WORK / "OpenKapitza")
t_dir = WORK / "tersoff_gpu"
sh(f"python3 scripts/prepare_cloud.py {t_dir} --potentials {WORK / 'lammps' / 'potentials'} --smoke "
   f"--only followup/sige_damp1.0_s2 --force")
wd = t_dir / "followup" / "sige_damp1.0_s2"; os.chdir(wd)
r = sh(f"{LMP} {KK} -in nemd.in -log log.lammps > stdout.log 2> stderr.log", check=False)
probs = run_smoke.check(wd); sp = run_smoke.speed(wd)
mac = 5 * 1.04e6                                          # M3 Pro, 5 ranks (resources.py)
if sp:
    print(f"Tersoff/kk: {sp['rate']:.3g} atom-steps/s, {sp['rate'] / mac:.1f}x the Mac (5 ranks);",
          "PASS" if not probs else "FAIL: " + "; ".join(probs))
SUMMARY["tersoff_gpu"] = {"ok": not probs, "problems": probs,
                          "atom_steps_per_s": sp["rate"] if sp else None}

In [ ]:
# 9. Summary: paste this back into the chat (also saved to Drive)
(CACHE / "colab_summary.json").write_text(json.dumps(SUMMARY, indent=1, default=str))
print(json.dumps(SUMMARY, indent=1, default=str))